### Model Comparison — Tuned
**Key fix:** Train models on **healthy data only** (anomaly == 0), then evaluate on a held-out set containing both normal and anomalous samples. Anomaly detectors learn what "normal" looks like — they can't do that if the training set is contaminated with failures.

## Setup

In [1]:
import sys
sys.path.insert(0, '..')

import warnings
warnings.filterwarnings('ignore', category=RuntimeWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
import logging
logging.basicConfig(level=logging.INFO)

from src.data_loader import load_cmapss, add_rul_to_train, create_anomaly_labels, get_sensor_columns
from src.preprocessing import (
    remove_constant_sensors, normalize_global,
    train_test_split_by_unit
)
from src.feature_engineering import build_feature_pipeline
from src.models import IsolationForestDetector, AutoencoderDetector, OneClassSVMDetector
from src.evaluation import evaluate_model, compare_models, find_optimal_threshold

## Prepare Data

In [2]:
train_df, _, _ = load_cmapss('FD001')
train_df = add_rul_to_train(train_df)
train_df = create_anomaly_labels(train_df, threshold=30)
sensor_cols = get_sensor_columns(train_df)
train_df, kept_sensors = remove_constant_sensors(train_df, sensor_cols)

# Feature engineering
featured_df = build_feature_pipeline(train_df, kept_sensors,
    rolling_windows=[5, 10], lags=[1, 5], ewma_spans=[5])

# Split by engine unit — 80% train engines, 20% test engines
train_split, test_split = train_test_split_by_unit(featured_df, test_ratio=0.2, seed=42)

print(f"Train engines: {train_split['unit_id'].nunique()}")
print(f"Test engines:  {test_split['unit_id'].nunique()}")
print(f"Train anomaly rate: {train_split['anomaly'].mean():.3f}")
print(f"Test anomaly rate:  {test_split['anomaly'].mean():.3f}")

INFO:src.data_loader:Loaded FD001: train=20631 rows, test=13096 rows, 100 engines
INFO:src.data_loader:Anomaly labels: 3100 anomalous (15.0%) out of 20631 samples
INFO:src.preprocessing:Dropped constant sensors: {'sensor_19', 'sensor_18', 'sensor_5', 'sensor_16', 'sensor_10', 'sensor_1'}
INFO:src.feature_engineering:Starting feature engineering pipeline...
INFO:src.feature_engineering:Added 60 rolling features
INFO:src.feature_engineering:Added 60 lag features
INFO:src.feature_engineering:Added 15 EWMA features
INFO:src.feature_engineering:Added 30 statistical features
INFO:src.feature_engineering:Final feature count: 188 columns
INFO:src.preprocessing:Split: 80 train units, 20 test units


Train engines: 80
Test engines:  20
Train anomaly rate: 0.152
Test anomaly rate:  0.144


## Prepare Feature Matrices

**Critical:** Separate healthy training data from the full set. Models train on healthy only.

We prepare two feature sets:
- **All 184 features** → for Isolation Forest and One-Class SVM (tree/kernel methods handle correlated features well)
- **15 raw sensor columns only** → for the Autoencoder (neural networks struggle with redundant engineered features)

In [3]:
exclude = ['unit_id', 'cycle', 'rul', 'anomaly']
feature_cols = [c for c in featured_df.columns if c not in exclude]

# Raw sensor columns only (for Autoencoder)
raw_sensor_cols = [c for c in kept_sensors]
print(f"All feature count:        {len(feature_cols)}")
print(f"Raw sensor count:         {len(raw_sensor_cols)}")

# Normalize using training stats — all features
train_split, scaler_all = normalize_global(train_split, feature_cols, method='standard')
test_split, _ = normalize_global(test_split, feature_cols, method='standard', scaler=scaler_all)

# === KEY FIX: Separate healthy training data ===
train_healthy = train_split[train_split['anomaly'] == 0]

# Full feature matrices (for IF and SVM)
X_train_healthy = np.nan_to_num(train_healthy[feature_cols].values, 0)
X_test = np.nan_to_num(test_split[feature_cols].values, 0)
y_test = test_split['anomaly'].values

# Raw sensor matrices (for Autoencoder)
X_train_healthy_raw = np.nan_to_num(train_healthy[raw_sensor_cols].values, 0)
X_test_raw = np.nan_to_num(test_split[raw_sensor_cols].values, 0)

print(f"\nHealthy training samples: {X_train_healthy.shape[0]}")
print(f"Test samples:            {X_test.shape[0]}")
print(f"Test anomaly count:      {y_test.sum()} ({y_test.mean():.1%})")

All feature count:        184
Raw sensor count:         15

Healthy training samples: 13860
Test samples:            4291
Test anomaly count:      620 (14.4%)


## Train Isolation Forest
Trained on healthy data only with all 184 features. Low contamination since the training set is clean.

In [4]:
iso_forest = IsolationForestDetector(
    contamination=0.05,
    n_estimators=300,
    random_state=42
)
iso_forest.fit(X_train_healthy)

iso_scores = iso_forest.score_samples(X_test)
iso_threshold = find_optimal_threshold(y_test, iso_scores)
iso_preds = (iso_scores > iso_threshold).astype(int)
iso_result = evaluate_model("Isolation Forest", y_test, iso_preds, iso_scores)
print(f"Optimal threshold: {iso_threshold:.4f}")

INFO:src.models.isolation_forest:Fitting Isolation Forest on 13860 samples...
INFO:src.evaluation:Isolation Forest: P=0.734 R=0.826 F1=0.777 AUC-PR=0.701 AUC-ROC=0.957


Optimal threshold: 0.4895


## Train Autoencoder
Trained on **raw sensor readings only** (15 features), not the full 184.

Why? Autoencoders learn to reconstruct inputs — when features are highly correlated (rolling means, lags of the same sensor), the reconstruction error becomes noisy and uninformative. Raw sensors give the autoencoder a clean signal to learn from.

In [ ]:
autoenc = AutoencoderDetector(
    input_dim=len(raw_sensor_cols),  # 15 raw sensors, not 184
    encoding_dim=8,                  # tight bottleneck for 15 features
    lr=1e-3,
    epochs=150,
    batch_size=256,
    threshold_percentile=95.0
)
autoenc.fit(X_train_healthy_raw)    # raw sensors only!

ae_scores = autoenc.score_samples(X_test_raw)
ae_threshold = find_optimal_threshold(y_test, ae_scores)
ae_preds = (ae_scores > ae_threshold).astype(int)
ae_result = evaluate_model("Autoencoder", y_test, ae_preds, ae_scores)
print(f"Optimal threshold: {ae_threshold:.6f}")

# Plot training loss
plt.figure(figsize=(8, 3))
plt.plot(autoenc.train_losses)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Autoencoder Training Loss (raw sensors)')
plt.tight_layout()
plt.show()

INFO:src.models.autoencoder:Training Autoencoder on 13860 samples (15 features), device=cpu
INFO:src.models.autoencoder:  Epoch 10/150, Loss: 0.177801
INFO:src.models.autoencoder:  Epoch 20/150, Loss: 0.153708
INFO:src.models.autoencoder:  Epoch 30/150, Loss: 0.147507
INFO:src.models.autoencoder:  Epoch 40/150, Loss: 0.143296
INFO:src.models.autoencoder:  Epoch 50/150, Loss: 0.139798
INFO:src.models.autoencoder:  Epoch 60/150, Loss: 0.138509
INFO:src.models.autoencoder:  Epoch 70/150, Loss: 0.136762
INFO:src.models.autoencoder:  Epoch 80/150, Loss: 0.136417
INFO:src.models.autoencoder:  Epoch 90/150, Loss: 0.134128
INFO:src.models.autoencoder:  Epoch 100/150, Loss: 0.133971
INFO:src.models.autoencoder:  Epoch 110/150, Loss: 0.132209
INFO:src.models.autoencoder:  Epoch 120/150, Loss: 0.132357


## Train One-Class SVM
Trained on healthy data only with all 184 features. Low nu since training data is clean.

In [ ]:
oc_svm = OneClassSVMDetector(
    kernel='rbf',
    gamma='scale',
    nu=0.05
)
oc_svm.fit(X_train_healthy)

svm_scores = oc_svm.score_samples(X_test)
svm_threshold = find_optimal_threshold(y_test, svm_scores)
svm_preds = (svm_scores > svm_threshold).astype(int)
svm_result = evaluate_model("One-Class SVM", y_test, svm_preds, svm_scores)
print(f"Optimal threshold: {svm_threshold:.4f}")

## Compare All Models

In [ ]:
results = [iso_result, ae_result, svm_result]
compare_models(results)

## Precision-Recall Curves

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for name, scores, auc_pr, color in [
    ("Isolation Forest", iso_scores, iso_result.auc_pr, '#4fc3f7'),
    ("Autoencoder", ae_scores, ae_result.auc_pr, '#ab47bc'),
    ("One-Class SVM", svm_scores, svm_result.auc_pr, '#ff7043')
]:
    prec, rec, _ = precision_recall_curve(y_test, scores)
    ax.plot(rec, prec, label=f'{name} (AUC-PR={auc_pr:.3f})', linewidth=2, color=color)

baseline = y_test.mean()
ax.axhline(y=baseline, color='gray', linestyle='--', alpha=0.5, label=f'Baseline ({baseline:.2f})')

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves (trained on healthy data only)')
ax.legend(loc='upper right')
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
plt.tight_layout()
plt.savefig('../data/model_comparison_pr.png', dpi=150, bbox_inches='tight')
plt.show()

## Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, result in zip(axes, results):
    cm = result.confusion_matrix
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(result.model_name, fontsize=12)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(['Normal', 'Anomaly'])
    ax.set_yticklabels(['Normal', 'Anomaly'])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i, j]), ha='center', va='center',
                    color='white' if cm[i, j] > cm.max()/2 else 'black', fontsize=14)

plt.tight_layout()
plt.savefig('../data/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

## Save Tuned Models

In [ ]:
import os
os.makedirs('../models', exist_ok=True)

iso_forest.save('../models/isolation_forest.pkl')
autoenc.save('../models/autoencoder.pt')
oc_svm.save('../models/one_class_svm.pkl')
print("All tuned models saved!")

## Summary

**What changed from the initial attempt:**
1. **Trained on healthy data only** — models learn "normal", not a mix of normal+failure
2. **Used optimal thresholds** from the PR curve (maximises F1) instead of default model thresholds
3. **Tuned hyperparameters** — lower contamination/nu since training data is clean, more trees/epochs for stability
4. **Autoencoder uses raw sensors only** — engineered features are highly correlated, which makes reconstruction error noisy and uninformative; raw sensors give the autoencoder a clean learning signal

**Design choice:** Isolation Forest and One-Class SVM use all 184 engineered features (tree/kernel methods handle correlation well), while the Autoencoder uses 15 raw sensor columns (neural networks need cleaner inputs).